In [1]:
"""
FinDER (FinanceRAG) retrieval evaluation harness — modular splitter x retriever sweep.
 
Loads FinDER corpus/queries/qrels from LOCAL files (Kaggle export), tries
multiple splitter types x chunk configs x retriever backends, and scores
every combination at multiple k values with Recall@k / NDCG@k.
 
Usage:
    pip install sentence-transformers datasets scikit-learn numpy pandas \
        langchain-text-splitters faiss-cpu chromadb rank_bm25 --break-system-packages
    python finder_splitter_eval.py
 
Not every dependency is required — retrievers/splitters with missing
libraries are skipped automatically (see the try/except ImportError guards).
"""
 
import math
from dataclasses import dataclass
try:
    from .retrievers import get_retriever
    from .splitters import get_splitter
except ImportError:
    from Retrievers import get_retriever   # falls back to a flat, non-package import
    from Splitters import get_splitter

import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder

"""The orchestrator. Knows the SHAPE of the sweep (splitter x config x
retriever x k) but not the concrete implementations — those are looked up
through the factories, so this file doesn't change when a new splitter or
retriever is added elsewhere."""

import numpy as np
import pandas as pd

D:\conda_envs\finder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import math
 
 
def dcg(rels: list[int]) -> float:
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))
 
 
def ndcg_at_k(retrieved_doc_ids: list[str], relevant_doc_ids: set[str], k: int) -> float:
    rels = [1 if d in relevant_doc_ids else 0 for d in retrieved_doc_ids[:k]]
    ideal_dcg = dcg(sorted(rels, reverse=True))
    return dcg(rels) / ideal_dcg if ideal_dcg > 0 else 0.0
 
 
def recall_at_k(retrieved_doc_ids: list[str], relevant_doc_ids: set[str], k: int) -> float:
    if not relevant_doc_ids:
        return 0.0
    hit = len(set(retrieved_doc_ids[:k]) & relevant_doc_ids)
    return hit / len(relevant_doc_ids)
    

In [3]:
"""Config objects — the single place that describes an experiment."""
 
from dataclasses import dataclass, field
 
 
@dataclass(frozen=True)
class SplitterSpec:
    """One splitter type + one (chunk_size, overlap) config to try."""
    name: str          # key into the splitter factory registry
    chunk_size: int
    overlap: int
 
 
@dataclass(frozen=True)
class DataConfig:
    corpus_path: str
    queries_path: str
    qrels_path: str
    qrels_cols: tuple[str, str, str] = ("query_id", "corpus_id", "score")
    qrels_delimiter: str = "\t"

@dataclass(frozen=True)
class ExperimentConfig:
    data: DataConfig
    embed_model: str
    splitter_specs: list[SplitterSpec]
    retriever_names: list[str]           # keys into the retriever factory registry
    k_values: list[int] = field(default_factory=lambda: [1, 3, 5, 10, 20])
    max_queries: int | None = 200
    max_corpus_docs: int | None = None
    output_csv: str = "../results/finder_hybridRetriever_results.csv"
    reranker_name: str | None = None            # e.g. "cross_encoder"; None = no reranking stage
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    rerank_pool: int = 50                        # candidates pulled from the retriever before reranking
    @property
    def max_k(self) -> int:
        return max(self.k_values)


 

In [4]:

class FinRAGDataLoader:
    def __init__(self, cfg: DataConfig):
        self.cfg = cfg
 
    def load(self):
        corpus_ds = load_dataset("json", data_files=self.cfg.corpus_path, split="train")
        queries_ds = load_dataset("json", data_files=self.cfg.queries_path, split="train")
        try:
            qrels_ds = load_dataset("csv", data_files=self.cfg.qrels_path,
                                     delimiter=self.cfg.qrels_delimiter, split="train")
        except Exception as e:
            raise SystemExit(f"Could not load qrels file '{self.cfg.qrels_path}': {e}")
        return list(corpus_ds), list(queries_ds), qrels_ds
 
    def build_relevant_map(self, qrels_ds, query_ids: set[str]) -> dict[str, set[str]]:
        qcol, dcol, scol = self.cfg.qrels_cols
        relevant: dict[str, set[str]] = {}
        for row in qrels_ds:
            qid = str(row[qcol])
            if qid not in query_ids:
                continue
            if row.get(scol, 1) and int(row[scol]) > 0:
                relevant.setdefault(qid, set()).add(str(row[dcol]))
        return relevant
        

In [11]:



class Pipeline:
    def __init__(self, cfg: ExperimentConfig):
        self.cfg = cfg
        self.loader = FinRAGDataLoader(cfg.data)
        self._model = SentenceTransformer(cfg.embed_model)
        # Load once, not per (splitter, retriever) combo — a cross-encoder is
        # comparatively expensive to load and is independent of chunking/retrieval choice.
        self.reranker = None

    def _build_chunks(self, corpus_rows, splitter_name, chunk_size, overlap):
        splitter = get_splitter(splitter_name, chunk_size, overlap)   # may raise ImportError; caller decides
        doc_ids, texts = [], []
        for row in corpus_rows:
            title = (row.get("title") or "").strip()
            body = row.get("text") or ""
            full_text = f"{title}\n{body}" if title else body
            for piece in splitter.split(full_text):
                if piece and piece.strip():
                    doc_ids.append(str(row["_id"]))
                    texts.append(piece)
        return doc_ids, texts

    def run(self) -> pd.DataFrame:
        cfg = self.cfg
        corpus_rows, queries_rows, qrels_ds = self.loader.load()
        if cfg.max_corpus_docs:
            corpus_rows = corpus_rows[:cfg.max_corpus_docs]
        if cfg.max_queries:
            queries_rows = queries_rows[:cfg.max_queries]

        query_ids = {str(r["_id"]) for r in queries_rows}
        relevant = self.loader.build_relevant_map(qrels_ds, query_ids)
        print(f"Corpus docs: {len(corpus_rows)} | Queries: {len(queries_rows)} "
              f"| Queries with qrels: {len(relevant)}")

        query_texts = [r["text"] for r in queries_rows]
        query_vecs = self._model.encode(query_texts, normalize_embeddings=True,batch_size=64, show_progress_bar=True )

        results = []
        for spec in cfg.splitter_specs:
            try:
                chunk_doc_ids, chunk_texts = self._build_chunks(corpus_rows, spec.name, spec.chunk_size, spec.overlap)
            except ImportError as e:
                print(f"Skipping splitter '{spec.name}': missing dependency ({e})")
                continue

            chunk_vecs = self._model.encode(chunk_texts)

            for retriever_name in cfg.retriever_names:
                try:
                    retriever = get_retriever(retriever_name)
                except KeyError as e:
                    print(e)
                    continue
                # If reranking, pull a wider candidate pool from the retriever first —
                # the retriever's own ranking within that pool no longer matters much,
                # since the reranker will reorder it before we score at each k.
                pool_k = max(cfg.max_k, cfg.rerank_pool) if self.reranker else cfg.max_k
                try:
                    retriever.fit(chunk_texts, chunk_vecs)
                    pool_idx = retriever.query(query_texts, query_vecs, pool_k)
                except ImportError as e:
                    print(f"Skipping retriever '{retriever_name}': missing dependency ({e})")
                    continue

                per_k_recall = {k: [] for k in cfg.k_values}
                per_k_ndcg = {k: [] for k in cfg.k_values}
                for qi, row in enumerate(queries_rows):
                    rel_docs = relevant.get(str(row["_id"]))
                    if not rel_docs:
                        continue
                    candidate_idx = list(pool_idx[qi])
                    if self.reranker:
                        candidate_texts = [chunk_texts[i] for i in candidate_idx]
                        order = self.reranker.rerank(row["text"], candidate_texts)
                        candidate_idx = [candidate_idx[o] for o in order]
                    retrieved = [chunk_doc_ids[i] for i in candidate_idx]
                    for k in cfg.k_values:
                        per_k_recall[k].append(recall_at_k(retrieved, rel_docs, k))
                        per_k_ndcg[k].append(ndcg_at_k(retrieved, rel_docs, k))

                for k in cfg.k_values:
                    results.append({
                        "splitter": spec.name, "chunk_size": spec.chunk_size, "overlap": spec.overlap,
                        "retriever": retriever_name,
                        "reranker": cfg.reranker_name or "none",
                        "k": k,
                        "n_chunks": len(chunk_texts),
                        "n_queries_scored": len(per_k_recall[k]),
                        "recall": np.mean(per_k_recall[k]) if per_k_recall[k] else float("nan"),
                        "ndcg": np.mean(per_k_ndcg[k]) if per_k_ndcg[k] else float("nan"),
                    })

        df = pd.DataFrame(results).sort_values(["k", "ndcg"], ascending=[True, False])
        df.to_csv(cfg.output_csv, index=False)
        return df

In [12]:
CONFIG = ExperimentConfig(
    data=DataConfig(
        corpus_path="../finrag_data/finder_corpus.jsonl/corpus.jsonl",
        queries_path="../finrag_data/finder_queries.jsonl/queries.jsonl",
        qrels_path="../finrag_data/FinDER_qrels.tsv",
    ),
    embed_model="sentence-transformers/all-MiniLM-L6-v2",
    splitter_specs=[
        SplitterSpec("char", 256, 40),
        SplitterSpec("char", 512, 0),
        SplitterSpec("recursive_char", 256, 64),
        SplitterSpec("token", 128, 20),
    ],
    retriever_names=["cosine_numpy", "faiss_flat", "bm25", "hybrid_rrf"],
    k_values=[1, 3, 5, 10, 20],
    max_queries=200,
    max_corpus_docs=None,
    # Optional second stage: reranks the top `rerank_pool` candidates from
    # each retriever above with a cross-encoder before scoring at each k.
    # Set to None to skip reranking entirely and go back to raw retriever output.
    reranker_name="cross_encoder",
    reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    rerank_pool=50,
)

In [13]:
if __name__ == "__main__":
    df = Pipeline(CONFIG).run()
    print(df.to_string(index=False))
    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3349.90it/s]


Corpus docs: 13867 | Queries: 200 | Queries with qrels: 61


Batches: 100%|██████████| 4/4 [00:00<00:00, 21.54it/s]


      splitter  chunk_size  overlap    retriever      reranker  k  n_chunks  n_queries_scored   recall     ndcg
          char         256       40 cosine_numpy cross_encoder  1     42451                61 0.150820 0.180328
          char         256       40   faiss_flat cross_encoder  1     42451                61 0.150820 0.180328
          char         512        0 cosine_numpy cross_encoder  1     23266                61 0.150820 0.180328
          char         512        0   faiss_flat cross_encoder  1     23266                61 0.150820 0.180328
recursive_char         256       64 cosine_numpy cross_encoder  1     53567                61 0.155738 0.163934
recursive_char         256       64   faiss_flat cross_encoder  1     53567                61 0.139344 0.147541
          char         256       40   hybrid_rrf cross_encoder  1     42451                61 0.097541 0.131148
         token         128       20 cosine_numpy cross_encoder  1     21450                61 0.109836 0